# 🪞 Lab 03 · When a planner fools itself, and how to measure honestly

**World Models course · Lectures 12, 21 and 23 · HW3 core and capstone evaluation** &nbsp;|&nbsp; ⏱ about 45 min &nbsp;|&nbsp; 💻 CPU only

Imagine asking a **flattering mirror** which outfit looks best. It is honest about most outfits, but it has one weird bug: it *loves* a particular ugly jacket. Try on 2 outfits and the mirror probably helps. Try on 1,000 and you will almost surely find that jacket, and the mirror will tell you it's perfect.

That is what happens when a planner searches hard inside an imperfect world model. In this lab you will:

1. Watch **more search make real results worse** while the model claims they are getting better.
2. Fix it with an **ensemble** that is cautious where its members disagree.
3. Learn to report results honestly with **bootstrap confidence intervals**.
4. See why you must resample **whole episodes**, not individual frames.

The same trap appears in *reward hacking* in LLM training, *best-of-N* sampling, and robot planners exploiting simulator bugs.

In [ ]:
#@title 🔧 Step 0 · Run this cell first (click ▶). It loads the tools for this lab. { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

# ---------------------------------------------------------------------------
# Guided-lab helpers. You never need to edit this cell.
#  * ___            : a blank for you to fill in
#  * check(name, x) : checks your answer; if it is blank or wrong, it explains
#                     and hands back a working version so the notebook keeps going
#  * quiz(id)       : a clickable multiple-choice question
#  * playground(...) : sliders that re-run a function when you let go
# ---------------------------------------------------------------------------
import inspect, html as _html
import numpy as np
from IPython.display import display, HTML
import os
try:
    import ipywidgets as widgets
    _WIDGETS = not os.environ.get("GUIDE_NO_WIDGETS")
except Exception:
    _WIDGETS = False

class BlankNotFilled(Exception):
    pass

class _Blank:
    """The ___ placeholder. Any maths with it stops with a friendly message."""
    __array_ufunc__ = None
    def _stop(self, *args, **kwargs):
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = _stop
    __truediv__ = __rtruediv__ = __floordiv__ = __rfloordiv__ = _stop
    __pow__ = __rpow__ = __matmul__ = __rmatmul__ = __mod__ = __rmod__ = _stop
    __neg__ = __pos__ = __abs__ = __getitem__ = __call__ = __iter__ = _stop
    __lt__ = __le__ = __gt__ = __ge__ = __bool__ = __float__ = __int__ = __index__ = _stop
    __array__ = __len__ = _stop
    def __getattr__(self, name):
        if name.startswith('__'):
            raise AttributeError(name)
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    def __repr__(self):
        return "___"

___ = _Blank()
CHALLENGES, QUIZZES = {}, {}
_solved, _quiz_score = {}, {}

_STYLE = {
    "ok":   ("#e8f6ee", "#1b7a4b", "✅"),
    "wait": ("#fff5e0", "#9a5b00", "🧩"),
    "bad":  ("#fdecea", "#b3261e", "❌"),
    "info": ("#eaf1fb", "#245eb5", "💡"),
}

def card(kind, title, body=""):
    bg, fg, icon = _STYLE[kind]
    display(HTML(
        f'<div style="background:{bg};border-left:5px solid {fg};padding:10px 14px;'
        f'border-radius:6px;margin:6px 0;color:#1d2530;font-size:14px;line-height:1.5">'
        f'<b style="color:{fg}">{icon} {title}</b><div>{body}</div></div>'))

def _as_numpy(x):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        return [_as_numpy(v) for v in x]
    return x

def _same(a, b, tol):
    a, b = _as_numpy(a), _as_numpy(b)
    if isinstance(a, list) or isinstance(b, list):
        return isinstance(a, list) and isinstance(b, list) and len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    try:
        a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    except Exception:
        return a == b
    return a.shape == b.shape and np.allclose(a, b, atol=tol, rtol=tol)

def _has_blank(obj):
    if isinstance(obj, _Blank):
        return True
    if callable(obj):
        try:
            return "___" in inspect.getsource(obj)
        except Exception:
            return False
    return False

def check(name, answer):
    """Check a challenge. Returns your answer if it works, otherwise a working reference."""
    ch = CHALLENGES[name]
    ref = ch["reference"]
    title = ch.get("title", name)
    fallback = ("<br><i>For now the notebook will use a working version so every later cell still runs. "
                "Come back, fill it in, and re-run this cell.</i>")
    if _has_blank(answer):
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” is waiting for you", "Hint: " + ch["hint"] + fallback)
        return ref
    try:
        if "test" in ch:
            ok, message = ch["test"](answer)
        elif callable(ref):
            ok, message = True, ""
            for args in ch["cases"]:
                args = args if isinstance(args, tuple) else (args,)
                expected, got = ref(*args), answer(*args)
                if not _same(expected, got, ch.get("tol", 1e-6)):
                    ok = False
                    message = "For a test input your function gave a different result from the expected one."
                    break
        else:
            ok = _same(ref, answer, ch.get("tol", 1e-6))
            message = f"You entered <code>{_html.escape(repr(_as_numpy(answer)))}</code>."
    except BlankNotFilled:
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” still has a blank", "Hint: " + ch["hint"] + fallback)
        return ref
    except Exception as err:
        ok, message = False, f"Running your version raised <code>{_html.escape(type(err).__name__)}: {_html.escape(str(err))}</code>."
    if ok:
        _solved[name] = True
        card("ok", f"Challenge solved: {title}", ch.get("why", ""))
        return answer
    _solved[name] = False
    card("bad", f"Not quite yet: {title}", message + "<br>Hint: " + ch["hint"] + fallback)
    return ref

def quiz(qid):
    q = QUIZZES[qid]
    question = f'<div style="font-size:15px;margin:8px 0 4px"><b>{"🔮 Predict: " if q.get("predict") else "🤔 "}{q["q"]}</b></div>'
    if not _WIDGETS:
        options = "".join(f"<li>{_html.escape(o)}</li>" for o in q["options"])
        display(HTML(question + f"<ol type='A'>{options}</ol><details><summary>Answer</summary>"
                     f"{'ABCDEFG'[q['answer']]}. {q['explain']}</details>"))
        return
    out = widgets.Output()
    buttons = []
    def choose(i):
        def handler(_):
            _quiz_score.setdefault(qid, i == q["answer"])
            for j, b in enumerate(buttons):
                b.button_style = "success" if j == q["answer"] else ("danger" if j == i else "")
            with out:
                out.clear_output()
                if i == q["answer"]:
                    card("ok", "Yes!", q["explain"])
                else:
                    card("bad", "Not this one. Here is the reasoning:", q["explain"])
        return handler
    for i, option in enumerate(q["options"]):
        b = widgets.Button(description=f"{'ABCDEFG'[i]}. {option}", layout=widgets.Layout(width="auto", max_width="100%"))
        b.on_click(choose(i))
        buttons.append(b)
    display(HTML(question), widgets.VBox(buttons), out)

def playground(fn, **controls):
    """controls: name=(min, max, step, default) for sliders, or name=[option, ...] for a dropdown."""
    defaults, sliders = {}, {}
    for name, spec in controls.items():
        if isinstance(spec, list):
            defaults[name] = spec[0]
            if _WIDGETS:
                sliders[name] = widgets.Dropdown(options=spec, value=spec[0], description=name)
        else:
            lo, hi, step, value = spec
            defaults[name] = value
            if _WIDGETS:
                kind = widgets.IntSlider if all(isinstance(v, int) for v in spec) else widgets.FloatSlider
                sliders[name] = kind(min=lo, max=hi, step=step, value=value, description=name,
                                     continuous_update=False, style={"description_width": "initial"},
                                     layout=widgets.Layout(width="420px"))
    if _WIDGETS:
        ui = widgets.VBox(list(sliders.values()))
        out = widgets.interactive_output(fn, sliders)
        display(ui, out)
    else:
        fn(**defaults)

def progress_report():
    solved = sum(_solved.values()); total = len(CHALLENGES)
    right = sum(_quiz_score.values()); asked = len(_quiz_score)
    stars = "⭐" * solved + "☆" * (total - solved)
    body = f"Challenges solved yourself: <b>{solved} / {total}</b> {stars}<br>"
    body += f"Quiz questions right on the first click: <b>{right} / {asked}</b> (of {len(QUIZZES)} in this lab)"
    missing = [CHALLENGES[k].get('title', k) for k in CHALLENGES if not _solved.get(k)]
    if missing:
        body += "<br>Still worth a try: " + ", ".join(missing)
    card("info", "Your progress in this lab", body)

# ---- this lab's challenges and quizzes ----
CHALLENGES["choose"] = dict(title="Plan by trusting the model",
    reference=lambda model_scores, budget: int(np.argmax(model_scores[:budget])),
    cases=[(np.array([0.1, 0.9, 0.3, 2.0]), 3), (np.array([5., 1., 2.]), 1)],
    hint="Look only at the first <code>budget</code> candidates and return the index of the highest model score (<code>np.argmax</code>).",
    why="This is how every sampling planner works: generate candidates, score them with the model, keep the best. The danger is in the next section.")

CHALLENGES["pessimism"] = dict(title="Be cautious where the ensemble disagrees",
    reference=lambda mean, spread, caution: mean - caution * spread,
    cases=[(np.array([1.0, 0.5]), np.array([0.2, 0.0]), 2.0)],
    hint="Start from the average opinion, then <b>subtract</b> caution × disagreement.",
    why="Penalising disagreement is the heart of <b>pessimistic model-based RL</b> (MOPO, MOReL) and a common guard against reward-model over-optimisation.")

def _test_boot(fn):
    vals = np.r_[np.zeros(30), np.ones(10)]
    means = np.asarray(fn(vals, 4000, np.random.default_rng(0)))
    if means.shape != (4000,):
        return False, f"Expected 4000 bootstrap means, got shape {means.shape}."
    expected_sd = vals.std() / np.sqrt(len(vals))
    if means.std() < 0.5 * expected_sd:
        return False, "Your resampled means barely vary. Are you sampling <b>with replacement</b>? A permutation always has the same mean."
    ok = abs(means.mean() - vals.mean()) < 0.02 and abs(means.std() - expected_sd) < 0.3 * expected_sd
    return ok, "The spread of your bootstrap means does not look like resampling n values with replacement."
def _ref_boot(values, repeats, rng):
    n = len(values)
    return np.array([values[rng.integers(0, n, size=n)].mean() for _ in range(repeats)])
CHALLENGES["resample"] = dict(title="Resample with replacement", reference=_ref_boot, test=_test_boot,
    hint="Draw <code>n</code> random indices between 0 and n−1, where repeats are allowed: <code>rng.integers(0, n, size=n)</code>.",
    why="Each resample is a “parallel universe” test set. How much the mean wobbles across universes tells you how much to trust it.")

CHALLENGES["interval"] = dict(title="A 95% interval",
    reference=lambda means: np.quantile(means, [0.025, 0.975]),
    cases=[np.linspace(0, 1, 1001)],
    hint="Cut off the lowest 2.5% and the highest 2.5% of bootstrap means: <code>np.quantile(means, [0.025, 0.975])</code>.",
    why="The middle 95% of bootstrap means is a <b>percentile bootstrap confidence interval</b>. It is what the rliable library (Google DeepMind) reports for RL agents.")

CHALLENGES["episode_mean"] = dict(title="Average frames within each episode first",
    reference=lambda frame_errors: frame_errors.mean(axis=1),
    cases=[np.arange(12.).reshape(3, 4)],
    hint="<code>frame_errors</code> has shape (episodes, frames). Average along the <b>frames</b> axis (axis=1).",
    why="Frames inside one episode share the same start state, lighting and luck, so they are not independent evidence. Episodes are the unit you can resample.")

QUIZZES["curse"] = dict(predict=True, q="As the planner tries more and more candidates, the <b>real</b> score of the action it picks will…",
    options=["keep improving", "improve at first, then get worse", "stay the same"],
    answer=1, explain="With few candidates, search finds genuinely good actions. With many, it becomes more likely to find the model's bug, which looks amazing to the model and is bad in reality. This is the <b>optimiser's curse</b> (also Goodhart's law).")
QUIZZES["ensemble"] = dict(q="Why does an ensemble of models with <i>different</i> bugs help?",
    options=["Five models are five times more accurate everywhere", "A bug in one member makes the members disagree, and disagreement can be penalised", "Ensembles cannot have bugs"],
    answer=1, explain="Where the data is good, members agree. Where one hallucinates, the spread rises. Caveat: members trained on the same gaps can share a bug, so pessimism reduces this failure but does not guarantee safety.")
QUIZZES["includes_zero"] = dict(q="The 95% interval for “B minus A” is [−0.01, +0.05]. What can you conclude?",
    options=["B and A are exactly equal", "B is definitely better", "With this many episodes we can't confidently tell them apart"],
    answer=2, explain="An interval that includes 0 means the data are consistent with no difference <i>and</i> with a small improvement. The honest move is to say so, or collect more episodes.")
QUIZZES["frames"] = dict(predict=True, q="We have 40 episodes × 50 frames. Which bootstrap gives a <b>narrower</b> interval, and is that narrowness trustworthy?",
    options=["Resampling 2,000 frames: narrower and trustworthy", "Resampling 2,000 frames: narrower, but falsely confident", "Resampling 40 episodes: narrower"],
    answer=1, explain="Treating 2,000 correlated frames as independent pretends you have far more evidence than you do. The interval shrinks, but it is wrong. Resample episodes.")
print('✅ Setup complete. Scroll down and run the cells in order.')

---
## 1 · A world model with a bug 🐛

Candidate actions are numbers between −1 and 1.
* **Reality:** the best action is 0. `true_score(a) = −a²`
* **The model:** matches reality everywhere **except** a narrow bump near `a = 0.8`, where it hallucinates a great score. Real learned models have spots like this, usually where training data was thin.

In [ ]:
def true_score(a):
    return -a**2

def model_score(a, bug_height=1.0, bug_width=0.01, bug_at=0.8):
    return -a**2 + bug_height * np.exp(-(a - bug_at)**2 / (2 * bug_width**2))

a = np.linspace(-1, 1, 2001)
plt.figure(figsize=(7, 3.3))
plt.plot(a, true_score(a), lw=3, label="reality")
plt.plot(a, model_score(a), "--", label="world model")
plt.annotate("the model's bug:\nlooks amazing, is bad", xy=(0.8, 0.36), xytext=(0.05, 0.25), arrowprops=dict(arrowstyle="->"))
plt.xlabel("candidate action"); plt.ylabel("score (higher is better)"); plt.legend(); plt.show()

### 🧩 Challenge 1 · Plan by trusting the model

In [ ]:
def choose(model_scores, budget):
    return ___    # 🧩 best of the first `budget` candidates, by the MODEL

choose = check("choose", choose)

<details><summary>🤔 <b>Need a hint?</b></summary>

Slice the first <code>budget</code> scores, then find the index of the largest.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return int(np.argmax(model_scores[:budget]))    # 🧩 best of the first `budget` candidates, by the MODEL</pre>

</details>

In [ ]:
quiz("curse")

---
## 2 · More search, worse reality 📉

For each budget, we run 400 independent planning problems. Each problem gets one pool of 1,024 random candidates, and a budget of *b* means *"look at the first b"*. Bigger budgets see a **superset** of the candidates, so search effort is the only thing that changes.

In [ ]:
rng = np.random.default_rng(3)
budgets = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
trials = 400
pool = rng.uniform(-1, 1, (trials, 1024))      # the same candidates for every budget

def search_curve(score_fn):
    believed, real = [], []
    scores = score_fn(pool)
    for b in budgets:
        picks = np.array([pool[t, choose(scores[t], b)] for t in range(trials)])
        believed.append(score_fn(picks).mean())      # what the model THINKS it achieved
        real.append(true_score(picks).mean())        # what actually happens
    return np.array(believed), np.array(real)

believed, real = search_curve(model_score)
plt.figure(figsize=(7, 3.5))
plt.plot(budgets, believed, "o--", label="what the model believes")
plt.plot(budgets, real, "o-", lw=2, label="what really happens")
plt.xscale("log", base=2); plt.xlabel("number of candidates searched"); plt.ylabel("score of the chosen action")
plt.title("The optimiser's curse"); plt.legend(); plt.show()
best = budgets[int(np.argmax(real))]
print(f"Real score peaks at a budget of {best} candidates ({real.max():.2f}), then falls to {real[-1]:.2f}.")
print(f"Meanwhile the model's own estimate keeps rising to {believed[-1]:.2f}. The gap is pure self-deception.")

### 🎛️ Playground · Size of the bug
Make the bug wider (easier to stumble into) or taller (more tempting). What happens to the best budget?

In [ ]:
def bug_lab(bug_height=1.0, bug_width=0.01):
    fn = lambda x: model_score(x, bug_height, bug_width)
    believed, real = search_curve(fn)
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
    axs[0].plot(a, true_score(a), lw=3); axs[0].plot(a, fn(a), "--"); axs[0].set_title("reality vs model")
    axs[1].plot(budgets, believed, "o--", label="believed"); axs[1].plot(budgets, real, "o-", label="real")
    axs[1].set_xscale("log", base=2); axs[1].set_title(f"best budget: {budgets[int(np.argmax(real))]}"); axs[1].legend()
    plt.show()

playground(bug_lab, bug_height=(0.0, 2.0, 0.1, 1.0), bug_width=(0.002, 0.1, 0.002, 0.01))

---
## 3 · A fix: a cautious ensemble 🤝

Train **five** models instead of one. Each picks up its own bug in a different place. For every candidate we compute:
* `mean`: the average opinion,
* `spread`: how much the five disagree (standard deviation).

A cautious planner scores each candidate by **average minus caution × disagreement**.

In [ ]:
quiz("ensemble")

### 🧩 Challenge 2 · Be cautious where the ensemble disagrees

In [ ]:
def cautious_score(mean, spread, caution):
    return ___       # 🧩 penalise disagreement

cautious_score = check("pessimism", cautious_score)

bug_places = [0.8, -0.55, 0.3, -0.9, 0.62]                        # each member's private bug
def ensemble_scores(x):
    return np.stack([model_score(x, bug_at=c) for c in bug_places])   # shape (5, ...)

plt.figure(figsize=(7, 3.5))
_, single_real = search_curve(model_score)
plt.plot(budgets, single_real, "o-", label="one model")
for caution in [0.0, 1.0]:
    fn = lambda x, c=caution: cautious_score(ensemble_scores(x).mean(0), ensemble_scores(x).std(0), c)
    _, real_c = search_curve(fn)
    plt.plot(budgets, real_c, "o-", label=f"ensemble, caution={caution}")
    print(f"ensemble caution={caution}: real score at budget 1024 = {real_c[-1]:.3f}")
print(f"single model:            real score at budget 1024 = {single_real[-1]:.3f}")
plt.xscale("log", base=2); plt.xlabel("number of candidates searched"); plt.ylabel("real score")
plt.legend(); plt.title("Averaging dilutes bugs; caution removes them"); plt.show()

<details><summary>🤔 <b>Need a hint?</b></summary>

The average opinion, minus (caution × spread).

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return mean - caution * spread       # 🧩 penalise disagreement</pre>

</details>

---
## 4 · Honest scoreboards: bootstrap confidence intervals 📏

You test controller **A** and controller **B** on the same test episodes. B's average is a bit higher. **Is B really better, or did it get lucky?**

The **bootstrap** answers this without any formulas:
1. Pretend your test episodes are the whole world.
2. Draw a new test set of the same size **with replacement** (some episodes twice, some not at all).
3. Recompute the average. Repeat 2,000 times.
4. The middle 95% of those averages is your **confidence interval**.

### 🧩 Challenge 3 · Resample with replacement

In [ ]:
def bootstrap_means(values, repeats, rng):
    n = len(values)
    means = []
    for _ in range(repeats):
        idx = ___     # 🧩 n random indices, repeats allowed
        means.append(values[idx].mean())
    return np.array(means)

bootstrap_means = check("resample", bootstrap_means)

<details><summary>🤔 <b>Need a hint?</b></summary>

<code>rng.integers(low, high, size=...)</code> draws random whole numbers; <code>high</code> is excluded.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>idx = rng.integers(0, n, size=n)     # 🧩 n random indices, repeats allowed</pre>

</details>

### 🧩 Challenge 4 · A 95% interval

In [ ]:
def interval_95(means):
    return ___   # 🧩 the middle 95%

interval_95 = check("interval", interval_95)

<details><summary>🤔 <b>Need a hint?</b></summary>

Which quantiles leave 2.5% on each side?

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return np.quantile(means, [0.025, 0.975])   # 🧩 the middle 95%</pre>

</details>

### 🎛️ Playground · How many test episodes do you need?
We simulate A and B on the **same** episodes. Some episodes are simply harder for both. We bootstrap the **per-episode difference** B − A, which keeps the comparison paired.

In [ ]:
def compare_controllers(n_episodes=20, true_improvement=0.03, seed=0):
    r = np.random.default_rng(seed)
    difficulty = r.normal(0, 0.15, n_episodes)                       # shared by A and B
    A = 0.60 + difficulty + r.normal(0, 0.08, n_episodes)
    B = 0.60 + true_improvement + difficulty + r.normal(0, 0.08, n_episodes)
    diff = B - A                                                     # paired, episode by episode
    means = bootstrap_means(diff, 2000, np.random.default_rng(1))
    lo, hi = interval_95(means)
    plt.figure(figsize=(6, 2.8))
    plt.hist(means, bins=40, color="#9ecae1"); plt.axvline(0, c="k", lw=1)
    plt.axvspan(lo, hi, color="orange", alpha=0.25, label="95% interval")
    plt.title(f"B − A = {diff.mean():+.3f}   95% interval [{lo:+.3f}, {hi:+.3f}]"); plt.legend(); plt.show()
    verdict = "B looks better" if lo > 0 else ("A looks better" if hi < 0 else "can't tell yet, so collect more episodes")
    print("Verdict:", verdict)

playground(compare_controllers, n_episodes=(5, 300, 5, 20), true_improvement=(0.0, 0.1, 0.005, 0.03), seed=(0, 30, 1, 0))

In [ ]:
quiz("includes_zero")

---
## 5 · Frames are not independent evidence 🎬

A world model is evaluated on **40 test episodes × 50 frames**. Its error on a frame depends mostly on *which episode* it is (start state, how chaotic that episode is), plus a little frame-to-frame noise.

In [ ]:
quiz("frames")

### 🧩 Challenge 5 · Average frames within each episode first

In [ ]:
def per_episode(frame_errors):
    return ___       # 🧩 one number per episode

per_episode = check("episode_mean", per_episode)

r = np.random.default_rng(5)
episode_offset = r.normal(0, 0.5, (40, 1))                     # what episode you're in matters a lot
frame_errors = 1.0 + episode_offset + r.normal(0, 0.1, (40, 50))

wrong = interval_95(bootstrap_means(frame_errors.reshape(-1), 2000, np.random.default_rng(2)))
right = interval_95(bootstrap_means(per_episode(frame_errors), 2000, np.random.default_rng(2)))
print(f"resampling 2,000 frames : [{wrong[0]:.3f}, {wrong[1]:.3f}]  width {wrong[1] - wrong[0]:.3f}  ← falsely confident")
print(f"resampling 40 episodes  : [{right[0]:.3f}, {right[1]:.3f}]  width {right[1] - right[0]:.3f}  ← honest")

<details><summary>🤔 <b>Need a hint?</b></summary>

Rows are episodes, columns are frames. Collapse the columns.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return frame_errors.mean(axis=1)       # 🧩 one number per episode</pre>

</details>

---
## 6 · Recap and industry links 🏭

| You saw | Name | Where it bites in industry |
|---|---|---|
| More search → worse reality | optimiser's curse, Goodhart's law | **reward hacking** when LLMs are optimised against a learned reward model; robots exploiting simulator glitches |
| The model grades its own choice | selection bias | best-of-N sampling needs an **independent verifier** (DeepMind AlphaCode, AlphaProof) |
| Cautious ensemble | pessimism / uncertainty penalties | offline RL for robots and recommendation systems |
| Bootstrap interval | statistical evaluation | Google DeepMind's **rliable**; Toyota Research's rigorous robot policy evaluations |
| Episodes, not frames | the unit of independence | every capstone result table you will ever make |

### 🧪 HW3 / capstone habits
1. In the bug playground, set `bug_height=0`. Does the curse disappear? What does that tell you about the cause?
2. In the controller playground, set `true_improvement=0.01`. Roughly how many episodes until the verdict becomes stable?
3. For your capstone, write down **before testing**: the primary metric, the number of test episodes, and what interval would change your conclusion.

### 🗣️ Explain it back
Why can a planner's *own* estimate of its plan be systematically too optimistic, even if the model is unbiased on random actions?

In [ ]:
progress_report()